# First Passage Time via SDE Approach
**Problem:** Starting at $X_0 = 1$, at each discrete step $X_{t+1} = X_t \cdot U_t$ where $U_t \overset{\text{iid}}{\sim} \text{Uniform}(0,1)$. Find $E[T]$ where $T = \min\{t \geq 1 : X_t \leq 0.5\}$.

## Step 1 — Rewrite as an increment equation

We can write the dynamics as:
$$\Delta X_t = X_{t+1} - X_t = X_t(U_t - 1)$$

This looks like a discrete stochastic differential equation: the **relative increment** $\Delta X_t / X_t = U_t - 1$ is i.i.d. with:
$$E[U_t - 1] = -\tfrac{1}{2}, \qquad \text{Var}[U_t - 1] = \tfrac{1}{12}$$

This mirrors the structure of **Geometric Brownian Motion** $dX_t = \mu X_t\,dt + \sigma X_t\,dW_t$, but working in discrete time with $X_t > 0$ always. The natural next move is the same one Itô uses for GBM: **take logarithms**.

## Step 2 — Log transform: $Y_t = \ln X_t$

Define $Y_t = \ln X_t$. Then:
$$Y_{t+1} - Y_t = \ln X_{t+1} - \ln X_t = \ln\left(\frac{X_{t+1}}{X_t}\right) = \ln U_t$$

So $Y_t$ is a **random walk** with i.i.d. increments $\ln U_t$. Since $Y_0 = \ln 1 = 0$:
$$Y_t = \sum_{k=1}^t \ln U_k$$

The hitting condition $X_t \leq \tfrac{1}{2}$ becomes:
$$Y_t \leq \ln\tfrac{1}{2} = -\ln 2$$

So we need $T = \min\{t \geq 1 : Y_t \leq -\ln 2\}$, the **first passage time** of the random walk below $-\ln 2$.

## Step 3 — Identify the increment distribution

The increments are $\ln U_t$ where $U_t \sim \text{Uniform}(0,1)$. Let $Z_t = -\ln U_t > 0$.

The CDF of $Z_t$: for $z > 0$,
$$P(Z_t \leq z) = P(-\ln U_t \leq z) = P(U_t \geq e^{-z}) = 1 - e^{-z}$$

So $Z_t = -\ln U_t \sim \text{Exponential}(1)$, with $E[Z_t] = 1$ and $\text{Var}[Z_t] = 1$.

Therefore the random walk has:
$$Y_t = -\sum_{k=1}^t Z_k, \qquad Z_k \overset{\text{iid}}{\sim} \text{Exp}(1)$$

This is a **random walk with drift $-1$ and variance $1$ per step**.

In [1]:
import random, math

# Verify -ln(U) ~ Exp(1) by checking mean and variance
rng = random.Random(42)
samples = [-math.log(rng.random()) for _ in range(1_000_000)]
mean = sum(samples) / len(samples)
var = sum((s - mean)**2 for s in samples) / len(samples)
print(f"E[-ln U]   = {mean:.5f}  (theoretical: 1.0)")
print(f"Var[-ln U] = {var:.5f}  (theoretical: 1.0)")

E[-ln U]   = 0.99951  (theoretical: 1.0)
Var[-ln U] = 0.99780  (theoretical: 1.0)


## Step 4 — Continuous SDE approximation

The random walk $Y_t$ has drift $\mu = -1$ and variance $\sigma^2 = 1$ per unit time. In continuous time, by the **Functional CLT (Donsker's theorem)**, this converges to:
$$dY_t = \underbrace{-1}_{\mu}\, dt + \underbrace{1}_{\sigma}\, dW_t$$

This is **Brownian motion with drift**. Note the parallel with GBM: applying Itô's lemma to $Y = \ln X$ for GBM $dX = \mu X\,dt + \sigma X\,dW$ gives $dY = (\mu - \sigma^2/2)\,dt + \sigma\,dW$. Here $\mu - \sigma^2/2 = -1$ and $\sigma = 1$ because the log of a Uniform(0,1) has those exact moments.

We want the **first passage time** $\tau = \inf\{t > 0 : Y_t \leq -\ln 2\}$ starting from $Y_0 = 0$.

## Step 5 — Feynman–Kac ODE for $v(y) = E_y[\tau]$

Let $v(y) = E[\tau \mid Y_0 = y]$ for $y > -\ln 2$ (above the barrier). By **Dynkin's formula**, $v$ satisfies the **Kolmogorov backward equation**:
$$\mathcal{L}\,v(y) = -1$$

where $\mathcal{L}$ is the generator of $dY_t = -dt + dW_t$:
$$\mathcal{L} = -\frac{d}{dy} + \frac{1}{2}\frac{d^2}{dy^2}$$

So the ODE is:
$$\frac{1}{2}v''(y) - v'(y) = -1$$

with boundary conditions:
- **Absorbing barrier:** $v(-\ln 2) = 0$
- **Growth condition:** $v(y) \sim y + \ln 2$ as $y \to +\infty$ (by pure drift, time to hit $-\ln 2$ from $y$ grows linearly in $y$), ruling out $e^{2y}$ growth

## Step 6 — Solving the ODE

**Homogeneous solution:** $\frac{1}{2}v'' - v' = 0$ has characteristic equation $\frac{1}{2}r^2 - r = 0 \Rightarrow r = 0$ or $r = 2$:
$$v_h(y) = C_1 + C_2 e^{2y}$$

**Particular solution:** try $v_p = ay$, then $\frac{1}{2}(0) - a = -1 \Rightarrow a = 1$:
$$v_p(y) = y$$

**General solution:**
$$v(y) = C_1 + C_2 e^{2y} + y$$

**Apply boundary conditions:**

1. Growth condition: $v(y) \not\sim e^{2y}$ as $y\to\infty$, so $C_2 = 0$.

2. Absorbing barrier at $y = -\ln 2$:
$$v(-\ln 2) = C_1 + (-\ln 2) = 0 \implies C_1 = \ln 2$$

**Solution:**
$$\boxed{v(y) = y + \ln 2}$$

**SDE answer:** starting from $Y_0 = 0$ (i.e., $X_0 = 1$):
$$E[\tau] = v(0) = \ln 2 \approx 0.693$$

## Step 7 — Overshoot correction (discrete $\to$ exact)

The SDE gives $\ln 2$, but simulation shows $1 + \ln 2$. The gap is the **overshoot** — in discrete time, the walk doesn't land exactly on $-\ln 2$; it jumps past it.

Apply **Wald's identity** to $S_n = \sum_{k=1}^n Z_k$ (the negative of $Y_n$) with $Z_k \sim \text{Exp}(1)$:
$$E[S_T] = E[T] \cdot E[Z_1] = E[T] \cdot 1 = E[T]$$

At stopping time $T$, the sum has exceeded $\ln 2$ by an overshoot $O \geq 0$:
$$S_T = \ln 2 + O$$

So $E[T] = \ln 2 + E[O]$. What is $E[O]$?

Since $Z_k \sim \text{Exp}(1)$ is **memoryless**: at the moment of crossing, the residual life of the current step is again $\text{Exp}(1)$. By the **renewal theory residual life formula** for Exp(1), $E[O] = E[Z^2]/(2E[Z]) = 2/2 = 1$.

Therefore:
$$\boxed{E[T] = \ln 2 + E[O] = \ln 2 + 1 = 1 + \ln 2}$$

The SDE answer is the **continuous first-passage time** $\ln 2$; the exact answer adds back the expected overshoot of $1$.

## Step 8 — Monte Carlo Verification

In [2]:
import random, math

rng = random.Random(42)
N = 1_000_000

# Simulate discrete process directly
total_T = 0
for _ in range(N):
    x = 1.0
    t = 0
    while x > 0.5:
        x *= rng.random()
        t += 1
    total_T += t

e_T = total_T / N

print("--- Results ---")
print(f"Monte Carlo E[T]           = {e_T:.6f}")
print(f"SDE answer   ln(2)         = {math.log(2):.6f}")
print(f"Exact answer 1 + ln(2)     = {1 + math.log(2):.6f}")
print(f"Error vs exact             = {abs(e_T - (1 + math.log(2))):.6f}")

--- Results ---
Monte Carlo E[T]           = 1.693707
SDE answer   ln(2)         = 0.693147
Exact answer 1 + ln(2)     = 1.693147
Error vs exact             = 0.000560


In [3]:
# Verify overshoot = 1 directly
rng2 = random.Random(0)
overshoots = []
for _ in range(N):
    s = 0.0
    while s < math.log(2):
        s += -math.log(rng2.random())   # Exp(1) step
    overshoots.append(s - math.log(2))

e_overshoot = sum(overshoots) / len(overshoots)
print(f"E[overshoot] = {e_overshoot:.6f}  (theoretical: 1.0)")
print(f"This is exactly E[T] - ln(2) = {(1 + math.log(2)) - math.log(2):.6f}")

E[overshoot] = 1.001153  (theoretical: 1.0)
This is exactly E[T] - ln(2) = 1.000000


## Summary

| Step | Transformation | Result |
|---|---|---|
| Discrete dynamics | $\Delta X_t = X_t(U_t - 1)$ | Multiplicative random walk |
| Log transform | $Y_t = \ln X_t$, $Y_{t+1} - Y_t = \ln U_t$ | Additive random walk with drift $-1$ |
| Increment dist. | $-\ln U_t \sim \text{Exp}(1)$ | Drift $= -1$, variance $= 1$ per step |
| Continuous SDE | $dY_t = -dt + dW_t$ | BM with drift |
| Feynman–Kac ODE | $\frac{1}{2}v'' - v' = -1$, $v(-\ln 2) = 0$ | $v(y) = y + \ln 2$ |
| SDE first passage | $v(0) = \ln 2$ | Continuous answer |
| Overshoot (Wald) | $E[T] = \ln 2 + E[O]$, $E[O] = 1$ | **$E[T] = 1 + \ln 2 \approx 1.693$** |